# Lab 1 – Build a Rotisserie Chicken Planning Agent

## Scenario
You will create a **RotisseriePlannerAgent** in your Azure AI Foundry project that serves as a rotisserie chicken planning assistant for the **Zava** store.

**The agent's job:**
- Recommend how many chickens to cook
- Recommend when to cook them during the day
- Suggest real-time adjustments to reduce waste
- Use past sales (not orders) to guide recommendations

Prioritize lunch and dinner peaks, avoid overcooking early morning and late evening, and explain recommendations in clear, simple language for store associates.

**Data sources:**
- `chicken_hourly_store_sales.json` – hourly sales, waste logs, stockout events
- `chicken_daily_orders_financials.json` – 31 days of daily financials

## Prerequisites
- Azure AI Foundry project with a deployed model (e.g., `gpt-4o-mini`)
- Python 3.10+
- Azure CLI authenticated: `az login`

### Values you need
- `AZURE_FOUNDRY_PROJECT_ENDPOINT` – your Foundry project endpoint URL

## Step 1 — Install required packages

In [ ]:
%pip install azure-ai-projects --pre
%pip install azure-ai-agents --pre
%pip install azure-identity openai python-dotenv

## Step 2 — Configure environment variables

Create a `.env` file in this folder (or set these in your shell).

In [ ]:
# Create/overwrite .env with your values
# AZURE_FOUNDRY_PROJECT_ENDPOINT="<paste-your-project-endpoint-here>"
# AZURE_FOUNDRY_PROJECT_MODEL_ID="gpt-4o-mini"

## Step 3 — Load config and authenticate

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

project_endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_id = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o-mini")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("Connected to project endpoint:", project_endpoint)
print("Model:", model_id)

## Step 4 — Upload data files to a vector store

We create a vector store and upload the two JSON data files so the agent can search them.

In [ ]:
# Create vector store
vector_store = openai_client.vector_stores.create(name="zava_rotisserie_data")
print(f"Vector store created (id: {vector_store.id})")

# Upload both data files
data_files = [
    "../data/chicken_hourly_store_sales.json",
    "../data/chicken_daily_orders_financials.json"
]

for file_path in data_files:
    file = openai_client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store.id, file=open(file_path, "rb")
    )
    print(f"Uploaded {file_path} (id: {file.id})")

tools = [{"type": "file_search", "vector_store_ids": [vector_store.id]}]

## Step 5 — Create the RotisseriePlannerAgent

This creates a persistent agent in your Foundry project with file search capabilities.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

planner_instructions = """You are a rotisserie chicken planning assistant for the Zava store.
You are given daily and hourly sales data.

Your job is to:
- Recommend how many chickens to cook
- Recommend when to cook them during the day
- Suggest simple real-time adjustments to reduce waste
- Use past sales (not orders) to guide recommendations

Prioritize lunch and dinner peaks, avoid overcooking early morning and late evening,
and explain your recommendations in clear, simple language for store associates.

Consider the attached chicken_hourly_store_sales.json and
chicken_daily_orders_financials.json as authoritative sources of Zava store sales data.

When making recommendations:
1. Always state the day of the week — patterns differ by day
2. Break cooking into batches aligned to demand peaks
3. Flag waste risks and suggest mitigation
4. Reference specific data points from the sales history"""

planner_agent = project_client.agents.create_version(
    agent_name="RotisseriePlannerAgent",
    definition=PromptAgentDefinition(
        model=model_id,
        instructions=planner_instructions,
        tools=tools,
    ),
    description="Rotisserie chicken planning assistant that recommends cooking schedules based on Zava store sales data.",
)

print("Created RotisseriePlannerAgent:", getattr(planner_agent, 'name', None))

## Step 6 — Test the agent

Ask the agent its core question: *What should I cook tomorrow and when?*

In [ ]:
from IPython.display import display, Markdown

agent_name = getattr(planner_agent, "name", "RotisseriePlannerAgent")
question = "What should I cook tomorrow and when?"

response = openai_client.responses.create(
    input=question,
    extra_body={"agent_reference": {"type": "agent_reference", "name": agent_name}},
)

display(Markdown(response.output_text))

## Step 7 — Test with more questions

Try additional prompts to validate the agent's capabilities.

In [ ]:
test_questions = [
    "How many chickens did we waste last week and what caused it?",
    "It's 2pm and we still have 12 chickens from the lunch batch. What should I do?",
    "Saturday is expected to be rainy. Should I adjust my cooking plan?",
    "What's our best and worst day for sales? Why?",
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")
    resp = openai_client.responses.create(
        input=q,
        extra_body={"agent_reference": {"type": "agent_reference", "name": agent_name}},
    )
    display(Markdown(resp.output_text))

## Step 8 — Record your deliverables

Save these values — you will need them in Lab 2 and Lab 3:

| Value | Where to find it |
|-------|------------------|
| `AZURE_FOUNDRY_PROJECT_ENDPOINT` | Your `.env` file |
| `ROTISSERIE_PLANNER_AGENT_NAME` | Output of Step 5 above |
| Vector Store ID | Output of Step 4 above |

## Validation Checklist
- ✅ Connected to Foundry project from Python
- ✅ Uploaded sales data to vector store
- ✅ Created RotisseriePlannerAgent with file search
- ✅ Agent answers cooking schedule questions using real sales data
- ✅ Agent provides waste reduction suggestions